# TRAINING MODEL

In [66]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

In [67]:
df = pd.read_csv("./datasets/skin_cancer.csv")

In [68]:
y = df["SkinCancer"].values
X = df.drop(columns=["SkinCancer"])

In [69]:
for col in X.columns:
    if X[col].dtype == "object":
        X[col] = LabelEncoder().fit_transform(X[col])
        
X = X.values

In [70]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [71]:
class BayesianDecisionModel:
    def fit(self, X, y):
        self.classes = np.unique(y)
        self.mean = {}
        self.var = {}
        self.prior = {}

        for c in self.classes:
            Xc = X[y == c]
            self.mean[c] = Xc.mean(axis=0)
            self.var[c] = Xc.var(axis=0) + 1e-6
            self.prior[c] = len(Xc) / len(X)

        self.build_loss_matrix()

    loss = {
        0: {0:0, 1:100},
        1: {0:10, 1:0}
    }
    
    def gaussian(self, c, x):
        mean, var = self.mean[c], self.var[c]
        return np.exp(-(x-mean)**2 / (2*var)) / np.sqrt(2*np.pi*var)

    def posterior(self, x):
        post = {}
        for c in self.classes:
            post[c] = np.prod(self.gaussian(c, x)) * self.prior[c]
        Z = sum(post.values())
        for c in post:
            post[c] /= Z
        return post

    def predict(self, X):
        return np.array([max(self.posterior(x), key=self.posterior(x).get) for x in X])

    def bayesian_update(self, prior, likelihood):
        unnorm = prior * likelihood
        return unnorm / np.sum(unnorm)

    def EMV(self, posterior):
        risks = []
        for d in self.classes:
            risks.append(sum(self.loss[d][c]*posterior[c] for c in self.classes))
        return min(risks)

    def EVSI(self, X):
        EMV0 = self.EMV(self.prior)
        return EMV0 - self.EVwSI(X)

    def EVwSI(self, X):
        return np.mean([self.EMV(self.posterior(x)) for x in X])

    def build_loss_matrix(self):
        c0, c1 = self.classes
        self.loss = {
            c0: {c0:0,  c1:150},
            c1: {c0:15, c1:0}
        }
    
    def clinical_decision(self, posterior):
        threshold = 0.20
        if posterior[self.classes[1]] >= threshold:
            return self.classes[1]
        return self.classes[0]

In [72]:
model = BayesianDecisionModel()
model.fit(X_train, y_train)

In [73]:
pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))
print("EVSI:", model.EVSI(X_test))

Accuracy: 0.8233399521568505
EVSI: 9.08295141745175


# PREDIKSI

In [74]:
encoders = {}
X_cols = df.columns.drop("SkinCancer")

for col in X_cols:
    if df[col].dtype == "object":
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])
        encoders[col] = le

X = df[X_cols].values
y = df["SkinCancer"].values

In [ ]:
def predict_patient(patient):
    x = []
    for col in X_cols:
        val = patient[col]
        if col in encoders:
            val = encoders[col].transform([val])[0]
        x.append(val)
    x = np.array(x)
    posterior = model.posterior(x)
    decision = model.clinical_decision(posterior)
    return decision, posterior

In [76]:
patient = {    
    "HeartDisease": "No",
    "BMI": 18.13,
    "Smoking": "No",
    "AlcoholDrinking": "No",
    "Stroke": "No",
    "PhysicalHealth": 0.0,
    "MentalHealth": 0.0,
    "DiffWalking": "No",
    "Sex": "Male",
    "AgeCategory": "80 or older",
    "Race": "White",
    "Diabetic": "No",
    "PhysicalActivity": "Yes",
    "GenHealth": "Excellent",
    "SleepTime": 8.0,
    "Asthma": "No",
    "KidneyDisease": "No"
}

decision, posterior = predict_patient(patient)

print("=== HASIL DIAGNOSIS ===")
print("Keputusan Sistem :", decision)
print("Posterior :", posterior)

=== HASIL DIAGNOSIS ===
Keputusan Sistem : No
Posterior : {'No': np.float64(0.8595751910858058), 'Yes': np.float64(0.1404248089141941)}


# SAVE MODEL

In [80]:
import os
import joblib

os.makedirs("model", exist_ok=True)

joblib.dump(model, "model/bdt_skin_model.pkl")
joblib.dump(encoders, "model/encoders.pkl")
joblib.dump(X_cols, "model/feature_columns.pkl")

print("Model & encoder berhasil disimpan!")

Model & encoder berhasil disimpan!
